In [9]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import networkx as nx
import icecream as ic

In [18]:
ROOT_PATH = '/Users/wolffjoa/Library/CloudStorage/OneDrive-mh-hannover.de/data/epigenetic-graph-analyzer'
graph_path = os.path.join(ROOT_PATH, 'FMI_data/MD_data_graph.graphml')
partition_path = os.path.join(ROOT_PATH, 'FMI_data/MD_data_partitions.csv')
community_path = os.path.join(ROOT_PATH, 'FMI_data/MD_data_communities.csv')

TAD_path = os.path.join(ROOT_PATH, 'FMI_data/Hi-C/MD_domains.bed')
PC_1_path = os.path.join(ROOT_PATH, 'FMI_data/Hi-C/MD_1.bw')
PC_2_path = os.path.join(ROOT_PATH, 'FMI_data/Hi-C/MD_2.bw')



In [11]:
G = nx.read_graphml(graph_path)

In [15]:
# Load the first communities file
communities1 = pd.read_csv(community_path)

# Transform communities into a dictionary where Gene field is the key and Community the value
gene_name_to_partition_mapping_sample_1 = {row['Gene']: row['Community'] for _, row in communities1.iterrows()}
    

In [19]:
# Load the first partitions file
partitions1 = pd.read_csv(partition_path)

# Transform partitions into a list of lists
partition_sample_1 = [list(map(int, entry.split(','))) for entry in partitions1['Content']]


In [20]:
partition_sample_1

[[2, 26147],
 [25784, 25867, 4],
 [26462, 6],
 [27488, 805, 27494, 808, 809, 810, 27497, 27499, 18, 27543, 27419, 27487],
 [27850, 21],
 [28165, 23],
 [28753, 25, 28766, 28654],
 [29792, 824, 27, 29837, 29838],
 [30209, 30],
 [32, 30067, 30071, 30073, 30074, 830, 31],
 [31110,
  31112,
  31113,
  31114,
  31115,
  31116,
  31117,
  31118,
  31119,
  31120,
  31121,
  31122,
  31059,
  31060,
  31057,
  31124,
  31128,
  31134,
  31135,
  31136,
  31137,
  34,
  35,
  33,
  30437,
  31138,
  31144,
  31145,
  31050],
 [31300, 31322, 31320, 31321, 42],
 [31682, 44, 31686],
 [51, 31791],
 [32368, 32377, 52],
 [32203, 32276, 53, 32231],
 [858, 59, 35567],
 [35656, 860, 61, 35652],
 [34736, 34729, 861, 62, 34733],
 [866, 867, 868, 35733, 67, 35735, 35740],
 [72, 35734],
 [36218, 36084, 76],
 [36549, 36554, 36528, 36529, 36502, 89, 36538, 36508],
 [96, 37329, 37270, 37334, 37279],
 [99, 100, 905, 37171, 37172, 37173, 37174, 37175, 37176, 37179, 37180],
 [37376, 37377, 37379, 101, 37350],
 [3

In [45]:
len(partition_sample_1)

3520

In [23]:
G.nodes["2"]

{'name': 'Atp6v1h_1',
 'colorType': 'green',
 'position': 'chr1:5083079-5083139',
 'partition': 0}

In [30]:
G.nodes["805"]

{'name': 'Snord87_1',
 'colorType': 'green',
 'position': 'chr1:9942533-9942593',
 'partition': 3}

In [29]:
list(G.neighbors("805"))

['27494']

In [99]:
TADs_df = pd.read_csv(TAD_path, sep='\t', header=None, names=['Chromosome', 'Start', 'End', "Id", "score", "strand", ".", ',', "color"])[['Chromosome', 'Start', 'End']]
TADs_df['Start'] = TADs_df['Start'].astype(int)
TADs_df['End'] = TADs_df['End'].astype(int)
TADs_df['Chromosome'] = 'chr' + TADs_df['Chromosome']

In [100]:
TADs_pr = pr.PyRanges(TADs_df)

/Users/wolffjoa/opt/miniconda3/envs/ega/lib/python3.10/site-packages/pyranges/methods/init.py:45: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  return {k: v for k, v in df.groupby(grpby_key)}


In [101]:
TADs_pr

,Chromosome,Start,End
0,chr1,4775000,5300000
1,chr1,5300000,6150000
2,chr1,6150000,6875000
3,chr1,6875000,8250000
4,chr1,8250000,8750000
...,...,...,...
3716,chrX,162900000,163150000
3717,chrX,163150000,163525000
3718,chrX,163525000,166650000
3719,chrX,166650000,167250000


In [ ]:
# make binsized genome
# count for each interaction its strength at start and end bin
# run hicexplorers TAD calling algorithm but for maximums instead of minimum
# two concaternated maxima should be the border of a TAD

In [120]:
import pyranges as pr

tad_interaction_threshold = 10
bin_size = 10000
start_community = []
end_community = []
community_list = []
all_positions = []
for community in partition_sample_1:
    community_position_list = []
    for node in community:
        chrom, positions = G.nodes[str(node)]['position'].split(":")
        start, end = positions.split("-")
        community_position_list.append([chrom, int(start), int(end)])

    if len(community_position_list) >= tad_interaction_threshold:
        all_positions.append(community_position_list)
        start_community.append(min(community_position_list))
        end_community.append(max(community_position_list))
        community_list.append([min(community_position_list)[0], min(community_position_list)[1], max(community_position_list)[2]])
    # max_position = max(community_position_list)
    # print(min_position, max_position)

    # BEGIN: Convert start_community and end_community into PyRanges objects
start_community_pr = pr.PyRanges(pd.DataFrame(start_community, columns=["Chromosome", "Start", "End"]))
end_community_pr = pr.PyRanges(pd.DataFrame(end_community, columns=["Chromosome", "Start", "End"]))
community_pr = pr.PyRanges(pd.DataFrame(community_list, columns=["Chromosome", "Start", "End"]))
    # END:

/Users/wolffjoa/opt/miniconda3/envs/ega/lib/python3.10/site-packages/pyranges/methods/init.py:45: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  return {k: v for k, v in df.groupby(grpby_key)}
/Users/wolffjoa/opt/miniconda3/envs/ega/lib/python3.10/site-packages/pyranges/methods/init.py:45: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  return {k: v for k, v in df.groupby(grpby_key)}
/Users/wolffjoa/opt/miniconda3/envs/ega/lib/python3.10/site-packages/pyranges/methods/init.py:45: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=Fa

In [119]:
all_positions[0]

[['chr1', 5083079, 5083139], ['chr1', 5884351, 5896685]]

In [95]:
start_community_pr

,Chromosome,Start,End
0,chr1,5083079,5083139
1,chr1,4856126,4861117
2,chr1,6487266,6487326
3,chr1,9544824,9546917
4,chr1,10835861,10837554
...,...,...,...
3515,chrX,161904763,161909945
3516,chrX,162899737,162902826
3517,chrX,166479637,166479697
3518,chrX,166673175,166673758


In [109]:
community_pr

,Chromosome,Start,End
0,chr1,5083079,5896685
1,chr1,4856126,6214674
2,chr1,6487266,6859101
3,chr1,9544824,10038227
4,chr1,10835861,11414115
...,...,...,...
3515,chrX,161904763,162829415
3516,chrX,162899737,164076541
3517,chrX,166479637,167232295
3518,chrX,166673175,167209210


In [122]:
TADs_pr.join(community_pr).head(30)

,Chromosome,Start,End,Start_b,End_b
0,chr1,4775000,5300000,3004106,171559203
1,chr1,4775000,5300000,4497360,88261254
2,chr1,5300000,6150000,3004106,171559203
3,chr1,5300000,6150000,4497360,88261254
4,chr1,6150000,6875000,3004106,171559203
5,chr1,6150000,6875000,4497360,88261254
6,chr1,6875000,8250000,3004106,171559203
7,chr1,6875000,8250000,4497360,88261254
8,chr1,6875000,8250000,7088934,10110362
9,chr1,8250000,8750000,3004106,171559203


In [102]:
TAD_start_intersect = TADs_pr.intersect(start_community_pr)

In [105]:
TAD_end_intersect = TADs_pr.intersect(end_community_pr)

In [104]:
TAD_start_intersect

,Chromosome,Start,End
0,chr1,4856126,4861117
1,chr1,5083079,5083139
2,chr1,6487266,6487326
3,chr1,7088934,7088994
4,chr1,9544824,9546917
...,...,...,...
3520,chrX,164177233,164178041
3521,chrX,165326641,165327867
3522,chrX,166479637,166479697
3523,chrX,166673175,166673758


In [106]:
TAD_end_intersect

,Chromosome,Start,End
0,chr1,5884351,5896685
1,chr1,5906194,5914460
2,chr1,6214614,6214674
3,chr1,6858080,6859101
4,chr1,10038167,10038227
...,...,...,...
3507,chrX,164076481,164076541
3508,chrX,164373568,164373628
3509,chrX,166170424,166170484
3510,chrX,167209150,167209210
